In [ ]:
# Load the shared metric implementation from this notebook directory.
from pathlib import Path
import sys
_metric_dir = Path.cwd() if (Path.cwd() / "multipoint_metrics.py").is_file() else Path.cwd() / "evaluation" / "digital_core"
if not (_metric_dir / "multipoint_metrics.py").is_file():
    raise FileNotFoundError("Open this notebook from evaluation/digital_core or the GLOSSY root.")
sys.path.insert(0, str(_metric_dir))
import multipoint_metrics as mpc

# %%
import numpy as np
import matplotlib.pyplot as plt

TESTB_FOLDER = globals().get("TESTB_FOLDER", r"../../data/real_binary")
DIMENSION = globals().get("DIMENSION", (128, 128, 128))
DATA_TYPE = globals().get("DATA_TYPE", np.int8)

# Fixed lag h; n is scanned.
H_FIXED = int(globals().get("H_FIXED", 1))
N_VALUES = np.asarray(
    globals().get("N_VALUES", np.arange(1, min(DIMENSION) + 1)), dtype=int
)

# RAW coding: "nonzero", "equal", or "less_equal".
PORE_DEFINITION = globals().get("PORE_DEFINITION", "nonzero")
PORE_VALUE = globals().get("PORE_VALUE", 1)
PORE_THRESHOLD = globals().get("PORE_THRESHOLD", 0)
PLOT_INDIVIDUAL = bool(globals().get("PLOT_INDIVIDUAL", True))


# %%
def list_raw_files(folder):
    folder = Path(folder)
    files = sorted(set(folder.glob("*.raw")) | set(folder.glob("*.RAW")))
    if not files:
        raise FileNotFoundError(f"No RAW files found in {folder}")
    return files


def read_raw_volume(path, dimension=DIMENSION, dtype=DATA_TYPE):
    return mpc.read_raw_volume(path, dimension, dtype)


def make_pore_mask(volume):
    return mpc.make_pore_mask(volume, PORE_DEFINITION, PORE_VALUE, PORE_THRESHOLD)


mpc_fixed_h = mpc.mpc_fixed_h_numpy


def curve_fixed_h(mask, h, n_values, axis):
    n_values = np.unique(np.asarray(n_values, dtype=int))
    y = np.array([mpc_fixed_h(mask, n, h, axis) for n in n_values])
    valid = np.isfinite(y)
    n_values = n_values[valid]
    return (n_values - 1) * h, y[valid], n_values


# %%
raw_files = list_raw_files(TESTB_FOLDER)
axis_names = ("X", "Y", "Z")
curves = {axis: [] for axis in range(3)}
names = []

for path in raw_files:
    volume = read_raw_volume(path)
    mask = make_pore_mask(volume)
    names.append(path.name)
    print(f"{path.name}: porosity={mask.mean():.6f}")
    for axis in range(3):
        curves[axis].append(curve_fixed_h(mask, H_FIXED, N_VALUES, axis))


# %%
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), sharey=True, constrained_layout=True)
colors = ("#1f77b4", "#d62728", "#2ca02c")

for axis, ax in enumerate(axes):
    if PLOT_INDIVIDUAL:
        for name, (distance, y, n) in zip(names, curves[axis]):
            ax.plot(distance, y, color=colors[axis], alpha=0.25, linewidth=1)
    distance0 = curves[axis][0][0]
    rows = [y for distance, y, n in curves[axis] if np.array_equal(distance, distance0)]
    matrix = np.vstack(rows)
    mean_y = matrix.mean(axis=0)
    std_y = matrix.std(axis=0)
    ax.plot(distance0, mean_y, color=colors[axis], linewidth=2, label="mean")
    if len(rows) > 1:
        ax.fill_between(distance0, mean_y - std_y, mean_y + std_y,
                        color=colors[axis], alpha=0.15, label="+/- 1 std")
    ax.set_title(f"{axis_names[axis]} direction")
    ax.set_xlabel("Total span (n-1)*h (voxel)")
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.25)
    ax.legend()

axes[0].set_ylabel(f"MPC (fixed h={H_FIXED}, varying n)")
fig.suptitle(f"MPC: fixed h={H_FIXED}, varying n")
plt.show()


# %%
# Optional simple REV candidate for the first file and X direction.
BETA = float(globals().get("BETA", 0.01))
distance, y, n = curves[0][0]
delta = np.abs(np.diff(y))
stable = [i for i in range(len(delta)) if np.all(delta[i:] <= BETA)]
distance_rev = distance[stable[0]] if stable else None
print("REV candidate (simple tail criterion):", distance_rev)



In [ ]:
# Load the shared metric implementation from this notebook directory.
from pathlib import Path
import sys
_metric_dir = Path.cwd() if (Path.cwd() / "multipoint_metrics.py").is_file() else Path.cwd() / "evaluation" / "digital_core"
if not (_metric_dir / "multipoint_metrics.py").is_file():
    raise FileNotFoundError("Open this notebook from evaluation/digital_core or the GLOSSY root.")
sys.path.insert(0, str(_metric_dir))
import multipoint_metrics as mpc

# %%
import numpy as np
import matplotlib.pyplot as plt
import torch

TESTB_FOLDER = globals().get("TESTB_FOLDER", r"../../data/real_binary")
DIMENSION = globals().get("DIMENSION", (128, 128, 128))
DATA_TYPE = globals().get("DATA_TYPE", np.int8)

# Fixed lag h; n is scanned.
H_FIXED = int(globals().get("H_FIXED", 1))
N_VALUES = np.asarray(
    globals().get("N_VALUES", np.arange(1, min(DIMENSION) + 1)), dtype=int
)

# RAW coding: "nonzero", "equal", or "less_equal".
PORE_DEFINITION = globals().get("PORE_DEFINITION", "nonzero")
PORE_VALUE = globals().get("PORE_VALUE", 1)
PORE_THRESHOLD = globals().get("PORE_THRESHOLD", 0)
PLOT_INDIVIDUAL = bool(globals().get("PLOT_INDIVIDUAL", True))
# Aggregate multiple RAW files by their largest MPC value at each lag.
CURVE_STATISTIC = globals().get("CURVE_STATISTIC", "max")

# CUDA is required by default. To deliberately run the same code on CPU, set
# DEVICE = "cpu" before running this script.
DEVICE = torch.device(globals().get("DEVICE", "cuda"))
if DEVICE.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Install a CUDA-enabled PyTorch build, or set DEVICE = 'cpu'."
    )
print(f"MPC compute device: {DEVICE}")


# %%
def list_raw_files(folder):
    folder = Path(folder)
    files = sorted(set(folder.glob("*.raw")) | set(folder.glob("*.RAW")))
    if not files:
        raise FileNotFoundError(f"No RAW files found in {folder}")
    return files


def read_raw_volume(path, dimension=DIMENSION, dtype=DATA_TYPE):
    return mpc.read_raw_volume(path, dimension, dtype)


def make_pore_mask(volume):
    return torch.from_numpy(mpc.make_pore_mask(volume, PORE_DEFINITION, PORE_VALUE, PORE_THRESHOLD)).to(device=DEVICE, dtype=torch.bool)


mpc_fixed_h = torch.inference_mode()(mpc.mpc_fixed_h_torch)


def curve_fixed_h(mask, h, n_values, axis):
    n_values = np.unique(np.asarray(n_values, dtype=int))
    y = np.array([mpc_fixed_h(mask, n, h, axis) for n in n_values])
    valid = np.isfinite(y)
    n_values = n_values[valid]
    return (n_values - 1) * h, y[valid], n_values


# %%
raw_files = list_raw_files(TESTB_FOLDER)
axis_names = ("X", "Y", "Z")
curves = {axis: [] for axis in range(3)}
names = []

for path in raw_files:
    volume = read_raw_volume(path)
    mask = make_pore_mask(volume)
    names.append(path.name)
    print(f"{path.name}: porosity={mask.float().mean().item():.6f}")
    for axis in range(3):
        curves[axis].append(curve_fixed_h(mask, H_FIXED, N_VALUES, axis))
    del volume, mask


# %%
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), sharey=True, constrained_layout=True)
colors = ("#1f77b4", "#d62728", "#2ca02c")

for axis, ax in enumerate(axes):
    if PLOT_INDIVIDUAL:
        for name, (distance, y, n) in zip(names, curves[axis]):
            ax.plot(distance, y, color=colors[axis], alpha=0.25, linewidth=1)
    distance0 = curves[axis][0][0]
    rows = [y for distance, y, n in curves[axis] if np.array_equal(distance, distance0)]
    matrix = np.vstack(rows)
    if CURVE_STATISTIC == "max":
        aggregate_y = matrix.max(axis=0)
        aggregate_label = "maximum"
    elif CURVE_STATISTIC == "mean":
        aggregate_y = matrix.mean(axis=0)
        aggregate_label = "mean"
    else:
        raise ValueError("CURVE_STATISTIC must be 'max' or 'mean'")
    ax.plot(distance0, aggregate_y, color=colors[axis], linewidth=2,
            label=aggregate_label)
    ax.set_title(f"{axis_names[axis]} direction")
    ax.set_xlabel("Total span (n-1)*h (voxel)")
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.25)
    ax.legend()

axes[0].set_ylabel(f"MPC (fixed h={H_FIXED}, varying n)")
fig.suptitle(f"MPC: fixed h={H_FIXED}, varying n ({CURVE_STATISTIC})")
plt.show()


# %%
# Optional simple REV candidate for the first file and X direction.
BETA = float(globals().get("BETA", 0.01))
distance, y, n = curves[0][0]
delta = np.abs(np.diff(y))
stable = [i for i in range(len(delta)) if np.all(delta[i:] <= BETA)]
distance_rev = distance[stable[0]] if stable else None
print("REV candidate (simple tail criterion):", distance_rev)


In [ ]:
# Load the shared metric implementation from this notebook directory.
from pathlib import Path
import sys
_metric_dir = Path.cwd() if (Path.cwd() / "multipoint_metrics.py").is_file() else Path.cwd() / "evaluation" / "digital_core"
if not (_metric_dir / "multipoint_metrics.py").is_file():
    raise FileNotFoundError("Open this notebook from evaluation/digital_core or the GLOSSY root.")
sys.path.insert(0, str(_metric_dir))
import multipoint_metrics as mpc

# %%
import numpy as np
import matplotlib.pyplot as plt
import torch

TESTB_FOLDER = globals().get("TESTB_FOLDER", r"../../data/real_binary_256")
DIMENSION = globals().get("DIMENSION", (256, 256, 256))
DATA_TYPE = globals().get("DATA_TYPE", np.int8)

# Fixed lag h; n is scanned.
H_FIXED = int(globals().get("H_FIXED", 1))
N_VALUES = np.asarray(
    globals().get("N_VALUES", np.arange(1, min(DIMENSION) + 1)), dtype=int
)

# RAW coding: "nonzero", "equal", or "less_equal".
PORE_DEFINITION = globals().get("PORE_DEFINITION", "nonzero")
PORE_VALUE = globals().get("PORE_VALUE", 1)
PORE_THRESHOLD = globals().get("PORE_THRESHOLD", 0)
PLOT_INDIVIDUAL = bool(globals().get("PLOT_INDIVIDUAL", True))
# Aggregate multiple RAW files by their largest MPC value at each lag.
CURVE_STATISTIC = globals().get("CURVE_STATISTIC", "max")

# CUDA is required by default. To deliberately run the same code on CPU, set
# DEVICE = "cpu" before running this script.
DEVICE = torch.device(globals().get("DEVICE", "cuda"))
if DEVICE.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Install a CUDA-enabled PyTorch build, or set DEVICE = 'cpu'."
    )
print(f"MPC compute device: {DEVICE}")


# %%
def list_raw_files(folder):
    folder = Path(folder)
    files = sorted(set(folder.glob("*.raw")) | set(folder.glob("*.RAW")))
    if not files:
        raise FileNotFoundError(f"No RAW files found in {folder}")
    return files


def read_raw_volume(path, dimension=DIMENSION, dtype=DATA_TYPE):
    return mpc.read_raw_volume(path, dimension, dtype)


def make_pore_mask(volume):
    return torch.from_numpy(mpc.make_pore_mask(volume, PORE_DEFINITION, PORE_VALUE, PORE_THRESHOLD)).to(device=DEVICE, dtype=torch.bool)


mpc_fixed_h = torch.inference_mode()(mpc.mpc_fixed_h_torch)


def curve_fixed_h(mask, h, n_values, axis):
    n_values = np.unique(np.asarray(n_values, dtype=int))
    y = np.array([mpc_fixed_h(mask, n, h, axis) for n in n_values])
    valid = np.isfinite(y)
    n_values = n_values[valid]
    return (n_values - 1) * h, y[valid], n_values


# %%
raw_files = list_raw_files(TESTB_FOLDER)
axis_names = ("X", "Y", "Z")
curves = {axis: [] for axis in range(3)}
names = []

for path in raw_files:
    volume = read_raw_volume(path)
    mask = make_pore_mask(volume)
    names.append(path.name)
    print(f"{path.name}: porosity={mask.float().mean().item():.6f}")
    for axis in range(3):
        curves[axis].append(curve_fixed_h(mask, H_FIXED, N_VALUES, axis))
    del volume, mask


# %%
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), sharey=True, constrained_layout=True)
colors = ("#1f77b4", "#d62728", "#2ca02c")

for axis, ax in enumerate(axes):
    if PLOT_INDIVIDUAL:
        for name, (distance, y, n) in zip(names, curves[axis]):
            ax.plot(distance, y, color=colors[axis], alpha=0.25, linewidth=1)
    distance0 = curves[axis][0][0]
    rows = [y for distance, y, n in curves[axis] if np.array_equal(distance, distance0)]
    matrix = np.vstack(rows)
    if CURVE_STATISTIC == "max":
        aggregate_y = matrix.max(axis=0)
        aggregate_label = "maximum"
    elif CURVE_STATISTIC == "mean":
        aggregate_y = matrix.mean(axis=0)
        aggregate_label = "mean"
    else:
        raise ValueError("CURVE_STATISTIC must be 'max' or 'mean'")
    ax.plot(distance0, aggregate_y, color=colors[axis], linewidth=2,
            label=aggregate_label)
    ax.set_title(f"{axis_names[axis]} direction")
    ax.set_xlabel("Total span (n-1)*h (voxel)")
    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.25)
    ax.legend()

axes[0].set_ylabel(f"MPC (fixed h={H_FIXED}, varying n)")
fig.suptitle(f"MPC: fixed h={H_FIXED}, varying n ({CURVE_STATISTIC})")
plt.show()


# %%
# Optional simple REV candidate for the first file and X direction.
BETA = float(globals().get("BETA", 0.01))
distance, y, n = curves[0][0]
delta = np.abs(np.diff(y))
stable = [i for i in range(len(delta)) if np.all(delta[i:] <= BETA)]
distance_rev = distance[stable[0]] if stable else None
print("REV candidate (simple tail criterion):", distance_rev)


In [ ]:
# Load the shared metric implementation from this notebook directory.
from pathlib import Path
import sys
_metric_dir = Path.cwd() if (Path.cwd() / "multipoint_metrics.py").is_file() else Path.cwd() / "evaluation" / "digital_core"
if not (_metric_dir / "multipoint_metrics.py").is_file():
    raise FileNotFoundError("Open this notebook from evaluation/digital_core or the GLOSSY root.")
sys.path.insert(0, str(_metric_dir))
import multipoint_metrics as mpc

# %%
import glob
import os

import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
except ImportError as exc:
    raise ImportError("This version requires PyTorch.") from exc


# %%
# Only the TEST/TI folder is used in this version.
TESTB_FOLDER = globals().get("TESTB_FOLDER", r"../../data/real_binary")
DIMENSION = globals().get("DIMENSION", (128, 128, 128))
DATA_TYPE = globals().get("DATA_TYPE", np.int8)
PORE_VALUE = globals().get("PORE_VALUE", 1)

# Fixed h, varying n.
H_FIXED = int(globals().get("H_FIXED", 1))
N_VALUES = np.asarray(
    globals().get("N_VALUES", np.arange(1, min(DIMENSION) + 1)),
    dtype=int,
)

MAX_R = int(globals().get("MAX_R", max(0, min(DIMENSION) - 1)))
BETA = float(globals().get("BETA", 0.0001))
# True: after the REV location, all subsequent |delta MPC| values must be <= BETA.
REV_STABLE_TAIL = bool(globals().get("REV_STABLE_TAIL", True))
OUTPUT_DIR = Path(globals().get("OUTPUT_DIR", "MPC_test_results"))
FIGURE_DPI = int(globals().get("FIGURE_DPI", 600))
PLOT_INDIVIDUAL = bool(globals().get("PLOT_INDIVIDUAL", False))

# RAW coding: "equal", "nonzero", or "less_equal".
PORE_DEFINITION = globals().get("PORE_DEFINITION", "equal")
PORE_THRESHOLD = globals().get("PORE_THRESHOLD", 0)

DEVICE_NAME = globals().get("DEVICE", "cuda")
if DEVICE_NAME == "auto":
    DEVICE_NAME = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE_NAME.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Set DEVICE = 'cpu', or install CUDA-enabled PyTorch."
    )
DEVICE = torch.device(DEVICE_NAME)
print(f"MPC device: {DEVICE}")


# %%
def raw_files(folder):
    files = sorted(glob.glob(os.path.join(str(folder), "*.raw")))
    if not files:
        raise FileNotFoundError(f"No .raw files found in {folder}")
    return files


def read_raw(path):
    return mpc.read_raw(path, DIMENSION, DATA_TYPE)


def pore_mask_torch(volume):
    mask = mpc.make_pore_mask(volume, PORE_DEFINITION, PORE_VALUE, PORE_THRESHOLD)
    return torch.as_tensor(mask, dtype=torch.bool, device=DEVICE)


mpc_fixed_h = torch.inference_mode()(mpc.mpc_fixed_h_torch)


def mpc_curve(mask, h, n_values, axis):
    n_values = np.unique(np.asarray(n_values, dtype=int))
    values = np.asarray([mpc_fixed_h(mask, n, h, axis) for n in n_values])
    valid = np.isfinite(values)
    n_values = n_values[valid]
    return (n_values - 1) * h, values[valid], n_values[valid]


def find_rev(distance, curve, beta=BETA, stable_tail=REV_STABLE_TAIL):
    return mpc.find_rev(distance, curve, beta, stable_tail)


# %%
files = raw_files(TESTB_FOLDER)
axis_names = ("X", "Y", "Z")
file_curves = {axis: [] for axis in range(3)}
file_revs = {axis: [] for axis in range(3)}
file_names = []

for index, path in enumerate(files, start=1):
    print(f"({index}/{len(files)}) {os.path.basename(path)}")
    volume = read_raw(path)
    mask = pore_mask_torch(volume)
    file_names.append(os.path.basename(path))
    print(f"    porosity={mask.float().mean().item():.6f}")

    for axis in range(3):
        distance, values, n_used = mpc_curve(mask, H_FIXED, N_VALUES, axis)
        plot_mask = distance <= MAX_R
        distance = distance[plot_mask]
        values = values[plot_mask]
        file_curves[axis].append((distance, values, n_used[plot_mask]))
        file_revs[axis].append(find_rev(distance, values))

    del volume, mask
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


# %%
# For each direction, select the RAW curve with the largest REV.
selected = {}
for axis in range(3):
    distance0 = file_curves[axis][0][0]
    if not all(np.array_equal(distance0, item[0]) for item in file_curves[axis]):
        raise ValueError("RAW files produced different distance grids.")

    matrix = np.vstack([item[1] for item in file_curves[axis]])
    revs = file_revs[axis]
    valid_rev = [(i, rev) for i, rev in enumerate(revs) if rev is not None]

    if valid_rev:
        selected_index, selected_rev = max(valid_rev, key=lambda pair: pair[1])
        selected_curve = matrix[selected_index]
    else:
        selected_index, selected_rev = 0, None
        selected_curve = matrix[0]
        print(f"{axis_names[axis]}: no REV found for BETA={BETA}")

    selected[axis] = {
        "distance": distance0,
        "minimum": matrix.min(axis=0),
        "maximum": matrix.max(axis=0),
        "matrix": matrix,
        "selected_curve": selected_curve,
        "selected_index": selected_index,
        "selected_file": file_names[selected_index],
        "selected_rev": selected_rev,
        "all_revs": revs,
    }
    print(
        f"{axis_names[axis]}: max REV={selected_rev} voxel, "
        f"file={file_names[selected_index]}"
    )

valid_axis_revs = [
    item["selected_rev"] for item in selected.values()
    if item["selected_rev"] is not None
]
overall_max_rev = max(valid_axis_revs) if valid_axis_revs else None
print(f"Overall maximum REV across X/Y/Z: {overall_max_rev} voxel")


# %%
def plot_test_mpc(selected, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(
        1, 3, figsize=(13.2, 4.2), sharey=True, constrained_layout=True
    )

    for axis, ax in enumerate(axes):
        item = selected[axis]
        distance = item["distance"]
        minimum = item["minimum"]
        maximum = item["maximum"]
        selected_curve = item["selected_curve"]
        rev = item["selected_rev"]

        # All values below/around the maximum are represented by this blue envelope.
        ax.fill_between(
            distance,
            minimum,
            maximum,
            color="#6EA8FE",
            alpha=0.34,
            linewidth=0,
            label="lower values",
        )
        if PLOT_INDIVIDUAL:
            for row in item["matrix"]:
                ax.plot(distance, row, color="#6EA8FE", alpha=0.20, linewidth=0.7)

        ax.plot(
            distance,
            selected_curve,
            color="#0057B8",
            linewidth=1.9,
            zorder=4,
            label="curve with maximum REV",
        )

        if rev is not None:
            rev_index = int(np.flatnonzero(distance == rev)[0])
            ax.axvline(
                rev,
                color="black",
                linestyle="--",
                linewidth=1.25,
                zorder=5,
                label=f"REV = {rev:g} voxel",
            )
            ax.scatter(
                [rev],
                [selected_curve[rev_index]],
                color="black",
                s=19,
                zorder=6,
            )

        ax.set_title(f"{axis_names[axis]} direction")
        ax.set_xlabel(r"Total span, $(n-1)h$ (voxel)")
        if axis == 0:
            ax.set_ylabel(r"MPC")
        ax.set_xlim(0, float(distance[-1]))
        ax.set_ylim(bottom=0)
        ax.grid(True, alpha=0.35, linewidth=0.55)
        ax.minorticks_on()
        ax.tick_params(direction="out", length=3.5, width=0.8)
        ax.legend(loc="upper right", frameon=False, fontsize=8.8)

    fig.suptitle(
        f"TEST MPC: fixed h={H_FIXED}, maximum REV standard, "
        f"|delta MPC| <= {BETA:g}",
        fontsize=13,
    )
    stem = output_dir / f"MPC_TEST_fixed_h{H_FIXED}_maxREV"
    fig.savefig(stem.with_suffix(".png"), dpi=FIGURE_DPI, bbox_inches="tight")
    fig.savefig(stem.with_suffix(".pdf"), dpi=FIGURE_DPI, bbox_inches="tight")
    fig.savefig(stem.with_suffix(".svg"), dpi=FIGURE_DPI, bbox_inches="tight")
    plt.show()
    plt.close(fig)


plot_test_mpc(selected, OUTPUT_DIR)
print(f"Done. Figures saved to: {OUTPUT_DIR.resolve()}")



In [ ]:
# Load the shared metric implementation from this notebook directory.
from pathlib import Path
import sys
_metric_dir = Path.cwd() if (Path.cwd() / "multipoint_metrics.py").is_file() else Path.cwd() / "evaluation" / "digital_core"
if not (_metric_dir / "multipoint_metrics.py").is_file():
    raise FileNotFoundError("Open this notebook from evaluation/digital_core or the GLOSSY root.")
sys.path.insert(0, str(_metric_dir))
import multipoint_metrics as mpc

# %%
import glob
import os

import numpy as np
import matplotlib.pyplot as plt

try:
    import torch
except ImportError as exc:
    raise ImportError("This version requires PyTorch.") from exc


# %%
# Only the TEST/TI folder is used in this version.
TESTB_FOLDER = globals().get("TESTB_FOLDER", r"../../data/real_binary")
DIMENSION = globals().get("DIMENSION", (128, 128, 128))
DATA_TYPE = globals().get("DATA_TYPE", np.int8)
PORE_VALUE = globals().get("PORE_VALUE", 1)

# Fixed h, varying n.
H_FIXED = int(globals().get("H_FIXED", 1))
N_VALUES = np.asarray(
    globals().get("N_VALUES", np.arange(1, min(DIMENSION) + 1)),
    dtype=int,
)

MAX_R = int(globals().get("MAX_R", max(0, min(DIMENSION) - 1)))
# REV threshold: deliberately reset here so an old notebook BETA cannot persist.
BETA = 0.0001
# True: after the REV location, all subsequent |delta MPC| values must be <= BETA.
REV_STABLE_TAIL = bool(globals().get("REV_STABLE_TAIL", True))
OUTPUT_DIR = Path(globals().get("OUTPUT_DIR", "MPC_test_results"))
FIGURE_DPI = int(globals().get("FIGURE_DPI", 600))
PLOT_INDIVIDUAL = bool(globals().get("PLOT_INDIVIDUAL", False))

PLOT_RC = {
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 10.6,
    "axes.labelsize": 12.6,
    "axes.titlesize": 12.0,
    "axes.linewidth": 1.0,
    "axes.edgecolor": "#202020",
    "axes.labelcolor": "#181818",
    "xtick.labelsize": 10.4,
    "ytick.labelsize": 10.4,
    "xtick.color": "#202020",
    "ytick.color": "#202020",
    "legend.fontsize": 9.8,
    "legend.frameon": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "grid.color": "#DADADA",
    "grid.linewidth": 0.55,
    "grid.alpha": 0.62,
}

# RAW coding: "equal", "nonzero", or "less_equal".
PORE_DEFINITION = globals().get("PORE_DEFINITION", "equal")
PORE_THRESHOLD = globals().get("PORE_THRESHOLD", 0)

DEVICE_SETTING = globals().get("DEVICE", "cuda")
# A previous notebook cell may already have converted DEVICE to torch.device.
DEVICE_NAME = str(DEVICE_SETTING)
if DEVICE_NAME == "auto":
    DEVICE_NAME = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE_NAME.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. Set DEVICE = 'cpu', or install CUDA-enabled PyTorch."
    )
DEVICE = torch.device(DEVICE_NAME)
print(f"MPC device: {DEVICE}")


# %%
def raw_files(folder):
    files = sorted(glob.glob(os.path.join(str(folder), "*.raw")))
    if not files:
        raise FileNotFoundError(f"No .raw files found in {folder}")
    return files


def read_raw(path):
    return mpc.read_raw(path, DIMENSION, DATA_TYPE)


def pore_mask_torch(volume):
    mask = mpc.make_pore_mask(volume, PORE_DEFINITION, PORE_VALUE, PORE_THRESHOLD)
    return torch.as_tensor(mask, dtype=torch.bool, device=DEVICE)


mpc_fixed_h = torch.inference_mode()(mpc.mpc_fixed_h_torch)


def mpc_curve(mask, h, n_values, axis):
    n_values = np.unique(np.asarray(n_values, dtype=int))
    values = np.asarray([mpc_fixed_h(mask, n, h, axis) for n in n_values])
    valid = np.isfinite(values)
    n_values = n_values[valid]
    return (n_values - 1) * h, values[valid], n_values[valid]


def find_rev(distance, curve, beta=BETA, stable_tail=REV_STABLE_TAIL):
    return mpc.find_rev(distance, curve, beta, stable_tail)


# %%
files = raw_files(TESTB_FOLDER)
axis_names = ("X", "Y", "Z")
file_curves = {axis: [] for axis in range(3)}
file_revs = {axis: [] for axis in range(3)}
file_names = []

for index, path in enumerate(files, start=1):
    print(f"({index}/{len(files)}) {os.path.basename(path)}")
    volume = read_raw(path)
    mask = pore_mask_torch(volume)
    file_names.append(os.path.basename(path))
    print(f"    porosity={mask.float().mean().item():.6f}")

    for axis in range(3):
        distance, values, n_used = mpc_curve(mask, H_FIXED, N_VALUES, axis)
        plot_mask = distance <= MAX_R
        distance = distance[plot_mask]
        values = values[plot_mask]
        file_curves[axis].append((distance, values, n_used[plot_mask]))
        file_revs[axis].append(find_rev(distance, values))

    del volume, mask
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


# %%
# For each direction, select the RAW curve with the largest REV.
selected = {}
for axis in range(3):
    distance0 = file_curves[axis][0][0]
    if not all(np.array_equal(distance0, item[0]) for item in file_curves[axis]):
        raise ValueError("RAW files produced different distance grids.")

    matrix = np.vstack([item[1] for item in file_curves[axis]])
    revs = file_revs[axis]
    valid_rev = [(i, rev) for i, rev in enumerate(revs) if rev is not None]

    if valid_rev:
        selected_index, selected_rev = max(valid_rev, key=lambda pair: pair[1])
        selected_curve = matrix[selected_index]
    else:
        selected_index, selected_rev = 0, None
        selected_curve = matrix[0]
        print(f"{axis_names[axis]}: no REV found for BETA={BETA}")

    selected[axis] = {
        "distance": distance0,
        "minimum": matrix.min(axis=0),
        "maximum": matrix.max(axis=0),
        "matrix": matrix,
        "selected_curve": selected_curve,
        "selected_index": selected_index,
        "selected_file": file_names[selected_index],
        "selected_rev": selected_rev,
        "all_revs": revs,
    }
    print(
        f"{axis_names[axis]}: max REV={selected_rev} voxel, "
        f"file={file_names[selected_index]}"
    )

valid_axis_revs = [
    item["selected_rev"] for item in selected.values()
    if item["selected_rev"] is not None
]
overall_max_rev = max(valid_axis_revs) if valid_axis_revs else None
print(f"Overall maximum REV across X/Y/Z: {overall_max_rev} voxel")


# %%
def curve_yrange(minimum, maximum, pad=0.075):
    values = np.concatenate([np.asarray(minimum), np.asarray(maximum)])
    values = values[np.isfinite(values)]
    if values.size == 0:
        return 0.0, 1.0
    y_min, y_max = float(values.min()), float(values.max())
    span = max(y_max - y_min, 1e-6)
    return max(0.0, y_min - span * pad), y_max + span * pad


def style_axes(ax, distance):
    from matplotlib.ticker import AutoMinorLocator, MaxNLocator

    ax.set_xlim(0, float(distance[-1]))
    ax.xaxis.set_major_locator(MaxNLocator(nbins=7, integer=True))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=6))
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.grid(True, which="major", axis="both")
    ax.grid(False, which="minor")
    ax.tick_params(axis="both", which="major", direction="out", length=3.9, width=0.85, pad=4)
    ax.tick_params(axis="both", which="minor", direction="out", length=2.0, width=0.65)
    for spine in ax.spines.values():
        spine.set_color("#202020")
        spine.set_linewidth(1.0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def plot_test_mpc(selected, output_dir):
    """Save one 4x4-inch publication-style figure for each direction."""
    output_dir.mkdir(parents=True, exist_ok=True)
    axis_names = ("X", "Y", "Z")

    with plt.rc_context(PLOT_RC):
        for axis, axis_name in enumerate(axis_names):
            item = selected[axis]
            distance = item["distance"]
            minimum = item["minimum"]
            maximum = item["maximum"]
            selected_curve = item["selected_curve"]
            rev = item["selected_rev"]
            fig, ax = plt.subplots(figsize=(4, 4), dpi=FIGURE_DPI)

            ax.fill_between(
                distance, minimum, maximum,
                color="#6EA8FE", alpha=0.34, linewidth=0,
                label="lower values",
            )
            if PLOT_INDIVIDUAL:
                for row in item["matrix"]:
                    ax.plot(distance, row, color="#6EA8FE", alpha=0.20, linewidth=0.7)

            ax.plot(
                distance, selected_curve,
                color="#0057B8", linewidth=1.5, zorder=4,
                label="curve with maximum REV",
            )
            if rev is not None:
                rev_index = int(np.flatnonzero(distance == rev)[0])
                ax.axvline(
                    rev, color="black", linestyle="--", linewidth=1.1,
                    zorder=5, label=f"REV = {rev:g} voxel",
                )
                ax.scatter([rev], [selected_curve[rev_index]], color="black", s=16, zorder=6)

            ax.set_xlabel(r"Lag distance, $r$ (voxel)")
            ax.set_ylabel(r"MPC")
            style_axes(ax, distance)
            ax.set_ylim(*curve_yrange(minimum, maximum))
            ax.text(
                0.018, 0.965, f"{axis_name} direction",
                transform=ax.transAxes, ha="left", va="top",
                fontsize=11.2, color="#161616",
            )
            ax.legend(
                loc="upper right", frameon=False, handlelength=3.0,
                handletextpad=0.65, borderaxespad=0.2, labelspacing=0.42,
            )

            stem = output_dir / f"MPC_TEST_{axis_name}_fixed_h{H_FIXED}_maxREV"
            fig.subplots_adjust(left=0.105, right=0.985, bottom=0.12, top=0.965)
            fig.savefig(stem.with_suffix(".png"), dpi=FIGURE_DPI, bbox_inches="tight", pad_inches=0.035)
            fig.savefig(stem.with_suffix(".pdf"), dpi=FIGURE_DPI, bbox_inches="tight", pad_inches=0.035)
            fig.savefig(stem.with_suffix(".svg"), dpi=FIGURE_DPI, bbox_inches="tight", pad_inches=0.035)
            plt.show()
            plt.close(fig)


plot_test_mpc(selected, OUTPUT_DIR)
print(f"Done. Figures saved to: {OUTPUT_DIR.resolve()}")
